# Private workspace initialization

Colab uses a Google-managed VM. Use the local path if cloud processing is unacceptable. Opening this notebook does not mount Drive or grant access.


In [ ]:
import os
import sys

in_colab = "google.colab" in sys.modules
PROFILE = os.environ.get(
    "GENOME_EVIDENCE_PROFILE", "personal_drive" if in_colab else "synthetic_ci"
)
REPOSITORY_URL = "https://github.com/jcollins-bioinfo/genome-evidence.git"
REPOSITORY_REF = os.environ.get("GENOME_EVIDENCE_GIT_REF", "main")
WORKSPACE_ROOT = os.environ.get(
    "GENOME_EVIDENCE_WORKSPACE", "/content/drive/MyDrive/genome-evidence-private"
)
SUBJECT_ID = os.environ.get("GENOME_EVIDENCE_SUBJECT_ID", "subject-0001")

In [ ]:
import importlib
import importlib.metadata
import json
import subprocess
from hashlib import sha256
from pathlib import Path

if PROFILE not in {"personal_drive", "synthetic_ci"}:
    raise ValueError("PROFILE must be personal_drive or synthetic_ci")

CHECKOUT = Path("/content/genome-evidence-src")
if PROFILE == "personal_drive":
    if "google.colab" in sys.modules:
        from google.colab import drive

        drive.mount("/content/drive", force_remount=False)
    if CHECKOUT.exists():
        remote = subprocess.run(
            ["git", "-C", str(CHECKOUT), "remote", "get-url", "origin"],
            check=True,
            capture_output=True,
            text=True,
            timeout=30,
        ).stdout.strip()
        if remote != REPOSITORY_URL:
            raise RuntimeError("Unexpected checkout remote; move the checkout aside and rerun")
        dirty = subprocess.run(
            ["git", "-C", str(CHECKOUT), "status", "--porcelain"],
            check=True,
            capture_output=True,
            text=True,
            timeout=30,
        ).stdout
        if dirty:
            raise RuntimeError("Checkout is dirty; preserve or move it aside and rerun")
    else:
        subprocess.run(
            ["git", "clone", "--no-checkout", REPOSITORY_URL, str(CHECKOUT)],
            check=True,
            timeout=180,
        )
    subprocess.run(
        ["git", "-C", str(CHECKOUT), "fetch", "--force", "origin", REPOSITORY_REF],
        check=True,
        timeout=180,
    )
    RESOLVED_COMMIT = subprocess.run(
        ["git", "-C", str(CHECKOUT), "rev-parse", "--verify", "FETCH_HEAD^{commit}"],
        check=True,
        capture_output=True,
        text=True,
        timeout=30,
    ).stdout.strip()
    subprocess.run(
        ["git", "-C", str(CHECKOUT), "checkout", "--detach", RESOLVED_COMMIT],
        check=True,
        timeout=60,
    )
    previously_imported = sys.modules.get("genome_evidence")
    if previously_imported is not None:
        previous_file = Path(getattr(previously_imported, "__file__", "")).resolve()
        if not previous_file.is_relative_to(CHECKOUT.resolve()):
            raise RuntimeError(
                "genome_evidence was already imported elsewhere; restart the runtime"
            )
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--disable-pip-version-check",
            "-e",
            f"{CHECKOUT}[notebook]",
        ],
        check=True,
        timeout=600,
    )
    source_root = (CHECKOUT / "src").resolve()
    package_init = source_root / "genome_evidence" / "__init__.py"
    if not package_init.is_file():
        raise RuntimeError("Resolved checkout does not contain the genome_evidence package")
    source_path = str(source_root)
    if source_path not in sys.path:
        sys.path.insert(0, source_path)
    importlib.invalidate_caches()
else:
    RESOLVED_COMMIT = "installed-ci-package"

genome_evidence = importlib.import_module("genome_evidence")
PACKAGE_ORIGIN = Path(genome_evidence.__file__).resolve()
if PROFILE == "personal_drive" and not PACKAGE_ORIGIN.is_relative_to(CHECKOUT.resolve()):
    raise RuntimeError("genome_evidence import origin is outside the resolved checkout")
INSTALLED_VERSION = importlib.metadata.version("genome-evidence")
LOCK_SHA256 = (
    sha256((CHECKOUT / "uv.lock").read_bytes()).hexdigest() if PROFILE == "personal_drive" else None
)
SANITIZED_IMPORT_PATH = (
    str(PACKAGE_ORIGIN.relative_to(CHECKOUT))
    if PROFILE == "personal_drive"
    else "installed-ci-package"
)
BOOTSTRAP_STATUS = {
    "profile": PROFILE,
    "requested_ref": REPOSITORY_REF,
    "resolved_commit": RESOLVED_COMMIT,
    "version": INSTALLED_VERSION,
    "import_path": SANITIZED_IMPORT_PATH,
    "lock_sha256": LOCK_SHA256,
    "lock_equivalent": PROFILE != "personal_drive",
}
print(json.dumps(BOOTSTRAP_STATUS, sort_keys=True))

In [ ]:
from genome_evidence.workspace import WorkspaceConfig, initialize_workspace

if PROFILE == "personal_drive":
    workspace = initialize_workspace(Path(WORKSPACE_ROOT), WorkspaceConfig(subject_id=SUBJECT_ID))
else:
    assert PROFILE == "synthetic_ci"
    workspace = None

In [ ]:
from genome_evidence.workspace import import_23andme_source

if PROFILE == "personal_drive":
    inbox = workspace / "inputs/inbox/23andme"
    candidates = [path for path in inbox.iterdir() if path.is_file() and not path.is_symlink()]
    if len(candidates) == 0:
        print("Workspace valid; inbox is empty. Upload one source file, then rerun notebook 00.")
    elif len(candidates) == 1:
        imported = import_23andme_source(workspace, candidates[0], SUBJECT_ID)
        print({"workspace": "valid", "imported_content_id": imported.name})
    else:
        raise ValueError("Multiple inbox files found; set an explicit selection before importing")
else:
    print("synthetic_ci: private workspace and Drive were not accessed")